# Matelda Pipeline

As data-driven applications gain popularity, ensuring high data quality is a growing concern. This requirement involves not only the quality of primary data sources but also external data sources used for data enrichment purposes. Yet, data cleaning techniques are limited to treating one table at a time. A table-by-table application of such methods is cumbersome, because these methods either require previous knowledge about constraints or often require labor-intensive configurations and manual labeling for each individual table. As a result, they hardly scale beyond a few tables and miss the chance for optimizing the cleaning process. To tackle these issues, we introduce a novel semi-supervised error detection approach, Matelda, that organizes a given set of tables by folding their cells with regard to domain and quality similarity to facilitate user supervision. The idea is to identify groups of data cells across all tables that can benefit from the same user label. For this purpose, we identify a feature embedding that makes cell values comparable across many different tables. Experimental evaluations demonstrate that Matelda outperforms various configurations of existing single-table cleaning methodologies in cleaning multiple tables at a time, in particular when the ratio of labeling budget to number of tables is very low.

For more information about Matelda, we recommend you to read the [Paper](https://openproceedings.org/2025/conf/edbt/paper-98.pdf) and view the corresponding Code on [GitHub](https://github.com/LUH-DBS/Matelda).

# Initialization of Matelda

The initialization function in Matelda sets up the necessary configurations, directories, and variables, including paths for input data, output, logs, and results. It prepares the environment for the error detection process, ensuring all required directories exist and the configuration parameters are loaded correctly.

In [1]:
from pipeline_functions import *
import multiprocessing
import os 
import pandas as pd
import pickle
import ipywidgets as widgets
from IPython.display import display

In [2]:
!conda install ipywidgets -y

Channels:
 - defaults
 - conda-forge
Platform: linux-64
Solving environment: done

# All requested packages already installed.



In [30]:
import ipywidgets as widgets
from IPython.display import display, Javascript

# Helper function to convert snake_case to a full descriptive title.
def get_full_name(key):
    return " ".join(word.capitalize() for word in key.split("_"))

# Default configuration dictionary.
configs = {
    "EXPERIMENTS": {
        "labeling_budget": 27700,
        "exp_name": "test_edbt",
        "n_cores": 128,
        "save_mediate_res_on_disk": 1,
        "final_result_df": False,
    },
    "DIRECTORIES": {
        "sandbox_dir": "datasets",
        "tables_dir": "Quintet",
        "output_dir": "output_quintet/output_quintet",
        "results_dir": "results",
        "logs_dir": "logs",
        "aggregated_lake_path": "aggregated_lake",
        "dirty_files_name": "dirty.csv",
        "clean_files_name": "clean.csv",
    },
    "TABLE_GROUPING": {
        "tg_enabled": 1,
        "tg_res_available": 0,
        "tg_method": "bert",
    },
    "COLUMN_GROUPING": {
        "cg_enabled": 1,
        "cg_res_available": 0,
        "min_num_labes_per_col_cluster": 2,
        "cg_clustering_alg": "hac",
    },
    "CELL_GROUPING": {
        "cell_feature_generator_enabled": 1,
        "cell_clustering_alg": "km",
        "cell_clustering_res_available": 0,
        "classification_mode": 1,
        "labels_per_cell_group": 1,
    },
    "RAHA": {
        "save_results": False,
        "strategy_filtering": False,
        "error_detection_algorithms": "OD, RVD, RVD_orig",
    }
}

# Dictionary to hold widget references by section and key.
widget_config = {}

# Fixed layout definitions for labels and inputs.
label_layout = widgets.Layout(width='250px')
input_layout = widgets.Layout(width='200px')  # Fixed width for all input widgets

# Build table-like rows for each configuration parameter.
for section, params in configs.items():
    widget_config[section] = {}
    rows = []  # Each row is an HBox with a label and an input widget.
    for key, value in params.items():
        # Create a fixed-width label widget.
        label_text = get_full_name(key)
        label_widget = widgets.Label(value=label_text, layout=label_layout)
        
        # Create the input widget based on type.
        if isinstance(value, bool):
            input_widget = widgets.Checkbox(
                value=value,
                description="",
                layout=widgets.Layout(width='200px', margin='0')
            )
        elif isinstance(value, int):
            input_widget = widgets.IntText(value=value, layout=input_layout)
        elif isinstance(value, float):
            input_widget = widgets.FloatText(value=value, layout=input_layout)
        else:
            input_widget = widgets.Text(value=str(value), layout=input_layout)
        
        # Save a reference to the widget.
        widget_config[section][key] = input_widget
        
        # Create a row (HBox) with the label and input widget.
        row = widgets.HBox(
            [label_widget, input_widget],
            layout=widgets.Layout(width='100%', align_items='flex-start', margin='2px 0')
        )
        rows.append(row)
    
    # Group the rows for this section in a VBox.
    widget_config[section]['_box'] = widgets.VBox(rows, layout=widgets.Layout(width='100%'))

# Create an Accordion to group configuration sections.
sections = list(widget_config.keys())
accordion = widgets.Accordion(
    children=[widget_config[sec]['_box'] for sec in sections],
    layout=widgets.Layout(width='100%')
)
for idx, sec in enumerate(sections):
    accordion.set_title(idx, sec)

# Create an Output widget to display the loading/notification messages.
loading_output = widgets.Output()

# Create the Save/Initialize button.
save_button = widgets.Button(
    description="Save Config and Initialize",
    button_style="success",
    layout=widgets.Layout(width='auto', margin='10px 0')
)

# Create the Reset to Default button.
reset_button = widgets.Button(
    description="Reset to Default",
    button_style="warning",
    layout=widgets.Layout(width='auto', margin='10px 0 10px 10px')
)

# Define the click event for the Save/Initialize button.
def on_save_button_clicked(b):
    # Disable the save button to prevent multiple clicks.
    save_button.disabled = True
    
    # Display the loading message.
    loading_output.clear_output()
    with loading_output:
        print("Loading configuration, please wait...")
    
    # Gather the updated config values from the input widgets.
    new_config = {}
    for section in widget_config:
        new_config[section] = {}
        for key, widget_item in widget_config[section].items():
            if key == '_box':
                continue
            new_config[section][key] = widget_item.value

    execution = 0  # Example execution value; adjust as needed.
    updated_configs = init(new_config, execution)  # Ensure your 'init' function is defined.
    
    # Clear the loading message and show the success message.
    loading_output.clear_output()
    with loading_output:
        print("Configuration saved and initialized!")
    
    # Re-enable the save button.
    save_button.disabled = False
    
    # Attempt to run the next cell in JupyterLab; if not available, show a reminder alert.
    display(Javascript('''
        if (window.jupyterapp && window.jupyterapp.commands) {
            window.jupyterapp.commands.execute("notebook:run-cell-and-select-next");
        } //else {
            alert("Continue with the next cell please.");
        }
    '''))

save_button.on_click(on_save_button_clicked)

# Define the click event for the Reset to Default button.
def on_reset_button_clicked(b):
    for section, params in configs.items():
        for key, default_value in params.items():
            widget_config[section][key].value = default_value
    loading_output.clear_output()
    with loading_output:
        print("Configuration reset to default.")

reset_button.on_click(on_reset_button_clicked)

# Create a header.
header = widgets.HTML("<h2>Configuration and Initialization</h2>")

# Wrap the buttons in an HBox to align them to the left.
button_box = widgets.HBox(
    [save_button, reset_button],
    layout=widgets.Layout(justify_content='flex-start')
)

# Display the header, the accordion, the buttons, and then the output messages.
display(header, accordion, button_box, loading_output)


HTML(value='<h2>Configuration and Initialization</h2>')

Accordion(children=(VBox(children=(HBox(children=(Label(value='Labeling Budget', layout=Layout(width='250px'))…

Output()

# Domain-Based Cell Folding
Domain-based cell folding in Matelda organizes cells from different tables by their semantic similarities.

In [6]:
# Set up multiprocessing pool (if necessary)
n_cores = configs["n_cores"]
pool = multiprocessing.Pool(n_cores)

# Call Domain Based Folding
table_grouping_dict, table_size_dict = domain_based_folding(configs, pool)

I need at least 2 labeled cells per table group to work at all and at least 2 * 6 labeled cells per table group to work effectively! Thant means you need to label 60 cells if you want reasonable (!) results:
{0: ['flights.csv'], 1: ['rayyan.csv'], 2: ['hospital.csv'], 3: ['beers.csv'], 4: ['movies_1.csv']}


### Results from Domain-Based Cell Folding

In [5]:
# Present the results in an interactive Widget
outer_acc = create_cell_fold_accordion(configs)
display(outer_acc)

Accordion(children=(Accordion(children=(Output(),), titles=('flights',)), Accordion(children=(Output(),), titl…

# Quality-Based Cell Folding

In [ ]:
# Use the keys returned by init (flattened names, not nested dictionaries)
cg_enabled = configs["column_grouping_enabled"]
column_groups_dir = os.path.join(configs["mediate_files_path"], "col_grouping_res", "col_df_res")
expected_file = os.path.join(column_groups_dir, "col_df_labels_cluster_0.pickle")

if cg_enabled and os.path.exists(expected_file):
    # Load the column grouping results to get the cluster sizes and the path to the column groups file.
    # This function returns a tuple of (number_of_col_clusters, cluster_sizes_dict, column_groups_df_path)
    _, cluster_sizes_dict, column_groups_df_path = loading_columns_grouping_results(
        table_grouping_dict, configs["mediate_files_path"]
    )

# Assuming column_groups_df_path and cluster_sizes_dict have been set by your domain-based folding cell:
if column_groups_df_path is not None:
    results = quality_based_folding(configs, pool, column_groups_df_path, cluster_sizes_dict)
    
    # Unpack the results
    (
        y_test_all, 
        y_local_cell_ids, 
        predicted_all, 
        y_labeled_by_user_all,
        unique_cells_local_index_collection, 
        samples, 
        n_user_labeled_cells
    ) = results
    
    # Present the results in the notebook
    print("Number of user labeled cells (quality based folding):", n_user_labeled_cells)
else:
    print("Skipping quality based folding due to missing column grouping results.")


### Presenting results from Quality-Based Cell Folding to user

In [ ]:
###############
# Problem:    #
###############
# - we discussed last time that I should look into the pickle files which are generated in error_detection
# - not sure what to visualize and what each column is
###############

# Load the DataFrame from your pickle file
df = pd.read_pickle('/home/julian/projects/Matelda/output_quintet/output_quintet_0/_test_edbt_Quintet_27700_labels/cell_clustering/all_cell_clusters_records.pickle')
#df = pd.read_pickle('/home/julian/projects/Matelda/output_quintet/output_quintet_0/_test_edbt_Quintet_27700_labels/cell_clustering/cell_cluster_cells_dict_all.pickle')
#columns_to_show = ['table_cluster', 'col_cluster', 'n_cells', 'cells_per_cluster']
#df_sample = df[columns_to_show].head(5)
df_sample = df.head(5)
df_sample

In [ ]:
####################
# Test             #
####################
# - tested the ipywidgets expand/collapse buttons
# - but thats probably not what we want to visualize
####################

# Load the DataFrame from your pickle file.
df = pd.read_pickle('/home/julian/projects/Matelda/output_quintet/output_quintet_0/_test_edbt_Quintet_27700_labels/cell_clustering/all_cell_clusters_records.pickle')

# Choose the columns you want to display and take a small sample.
columns_to_show = ['table_cluster', 'col_cluster', 'n_cells', 'cells_per_cluster']
df_sample = df[columns_to_show].head(5).copy()  # Using head(5) to keep it light.

def format_cells_per_cluster(cell_cluster):
    """
    Convert the cells_per_cluster dictionary into a multi-line string.
    For each key, show only the first 10 items of the list (if the list is long).
    """
    if isinstance(cell_cluster, dict):
        lines = []
        for key, value in cell_cluster.items():
            # Show a preview of the list; adjust the number (10 here) as needed.
            if isinstance(value, list) and len(value) > 10:
                preview = value[:10]
                lines.append(f"{key}: {preview} ... ({len(value)} items)")
            else:
                lines.append(f"{key}: {value}")
        return "\n".join(lines)
    return str(cell_cluster)

# Create an accordion widget for each row in the sample.
accordion_items = []
for idx, row in df_sample.iterrows():
    # Build a summary header for the accordion.
    header = f"Cluster: ({row['table_cluster']}, {row['col_cluster']}), n_cells: {row['n_cells']}"
    
    # Format the full details of cells_per_cluster.
    details = format_cells_per_cluster(row['cells_per_cluster'])
    
    # Create an output widget to hold the formatted details (using <pre> to preserve formatting).
    detail_output = widgets.HTML(value=f"<pre>{details}</pre>")
    
    # Create an Accordion widget with this detail.
    accordion = widgets.Accordion(children=[detail_output])
    accordion.set_title(0, header)
    
    accordion_items.append(accordion)

# Display all the accordions vertically.
display(widgets.VBox(accordion_items))